# Conditional Variational Autoencoder applied to CelebA Dataset

The CVAE model was adapted from https://github.com/AntixK/PyTorch-VAE/.

The model was trained on the CelebA dataset: http://mmlab.ie.cuhk.edu.hk/projects/CelebA.html

Images were resized to 128x128 pixels.

Antonio Esteves @ UMinho, Mar 2024

## Import necessary libraries

In [ ]:
import os
from   pathlib  import Path
import pandas   as     pd
import torch
import torch.nn         as     nn
from   torch.nn         import functional as F
from   torch.utils.data import Dataset
from   typing   import List, TypeVar, Tuple, Dict, Union #  Callable, Any, Optional, Sequence
from   PIL      import Image
from   natsort  import natsorted

Tensor = TypeVar('torch.tensor')

## Import W&B and Login

In [ ]:
import wandb

wandb.login()

## Read the configuration file

In [ ]:
import yaml

BASE_FILE_NAME = "CVAE_CelebA_128x128_01"
CONFIG_FILE    = '../config/cvae_antixk_config2.yaml'

with open(CONFIG_FILE, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

In [ ]:
config

## Track metadata and hyperparameters with `wandb.init`

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = dict(
    epochs        = config["exp_params"]["epochs"],
    batch_size    = config["data_params"]["train_batch_size"],
    learning_rate = config["exp_params"]["LR"],
    kld_weight    = config["exp_params"]["kld_weight"],
    optimizer     = "Adam",
    dataset       = "CelebA",
    image_size    = [
                    config["model_params"]["in_channels"],
                    config["data_params"]["patch_size"],
                    config["data_params"]["patch_size"]
                    ],
    latent_dim     = config["model_params"]["latent_dim"],
    num_attributes = config["model_params"]["num_attributes"],
    architecture   = "VAE-cnn"
)

In [ ]:
config_wandb

In [ ]:
wandb.init(
    project = 'OUR_WANDB_PROJECT_ID',
    entity  = 'OUR_WANDB_ENTITY', 
    config  = config_wandb
)

## Initializations

In [ ]:
print(torch.__version__)

# setup device agnostic code

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

# Setup path to celebA dataset folder

data_path = Path(config["data_params"]["data_path"])

train_dir = data_path / "train"
val_dir   = data_path / "val"
test_dir  = data_path / "test"

# Attribute file path
attr_file = data_path / "list_attr_celeba.txt"


## Exploring the CelebA dataset

In [ ]:
# Get all training image paths
train_image_list = list(train_dir.glob("*.jpg")) 

print(f'Training set size: {len(train_image_list)}')

# Get all validation image paths
val_image_list = list(val_dir.glob("*.jpg")) 

print(f'Validation set size: {len(val_image_list)}')

# Get all test image paths
test_image_list = list(test_dir.glob("*.jpg"))

print(f'Test set size: {len(test_image_list)}')


In [ ]:
# Visualizing random images

import random
from   PIL import Image

# set seed

random.seed(42)

# Pick a random image path

rand_image_path = random.choice(train_image_list)
print(f'Randomly selected image:\n\t{rand_image_path}')

# Open the image using pillow library

img = Image.open(rand_image_path)

# Show the image and print image metadata

print(f'Random image path:   {rand_image_path}')
print(f'Random image height: {img.height}')
print(f'Random image width:  {img.width}')

display(img)

In [ ]:
# Visualize an image using matplotlib

import numpy as np
import matplotlib.pyplot as plt

# convert the image 'img' to a numpy array

img_array = np.asarray(img)

# plot the image with matplotlib

plt.figure(figsize=(10,7))
plt.imshow(img_array)
plt.title(f'Image shape: {img_array.shape} (height,width,channels)')
plt.axis(False)

## Transforming the Data

Before we can use our data with PyTorch:
1. Convert the images to tensors using `torchvision.transforms`([documentation](https://pytorch.org/vision/0.16//transforms.html))
2. Create a custom Dataset from the images in a folder and associated attributes using `torch.utils.data.Dataset`
3. Create training, validation and test datasets
4. Convert the `Datasets` into `torch.utils.data.DataLoaders`

### (i) Convert images to tensors

In [ ]:
from torch.utils.data import DataLoader
from torchvision      import transforms, datasets

train_transforms = transforms.Compose(
    [
    transforms.RandomHorizontalFlip(),
    transforms.CenterCrop(148),
    transforms.Resize(config["data_params"]["patch_size"]),
    transforms.ToTensor()
    ]
)

val_transforms = transforms.Compose(
    [
    transforms.RandomHorizontalFlip(),
    transforms.CenterCrop(148),
    transforms.Resize(config["data_params"]["patch_size"]),
    transforms.ToTensor()
    ]
)

In [ ]:
img_tensor = train_transforms(img)
print(img_tensor.shape)
print(img_tensor)

In [ ]:
# Visualize transformed images

def plot_transformed_images(
    image_paths,
    transform,
    n           = 3,
    seed        = None ):
  """
  The functions selects 'n' random images from 'image_paths', load and transform them,
  then it plots the original images vs the transformed representation.
  """
  # set the random seed if it is not None
  if seed:
    random.seed(seed)

  # randomly select 'n' image paths
  rand_image_paths = random.sample(image_paths,k=n)

  # iterate throuth the 'n' selected paths
  for image_path in rand_image_paths:

    with Image.open(image_path) as f:

      # original image
      fig,ax = plt.subplots(nrows=1,ncols=2)
      ax[0].imshow(f)
      ax[0].set_title(f'Original\nSize: {f.size}')
      ax[0].axis(False)

      # transform the original image and plot the transformed representation
      transformed_image = transform(f).permute(1, 2, 0) # we need to convert C,H,W (pytorch) to H,W,C (plt)
      ax[1].imshow(transformed_image)
      ax[1].set_title(f'Transformed\nShape: {transformed_image.shape}')
      ax[1].axis(False)

      fig.suptitle(f'Class: {image_path.parent.stem}',fontsize=16)

In [ ]:
plot_transformed_images(
    image_paths = train_image_list,
    transform   = train_transforms,
    n           = 3
)

### (ii) Create a custom Dataset from the images in a folder and the associated attributes using `torch.utils.data.Dataset`

In [ ]:
class CelebaDataset(Dataset):
    """
    Custom Dataset for loading CelebA face images and associated attributes.
    """

    def __init__(self, attr_file, image_dir, transform=None):

        # Read attributes from file to DataFrame
        attr_df = pd.read_csv(attr_file, sep="\s+", skiprows=1)

        # Get the attribute names
        attr_names = list(attr_df.columns)

        # Replace "-1" by "0" in all columns
        for name in attr_names:
            # dictionary of replacements
            replacements = {-1: 0}

            # replace values using the .map() method
            attr_df[name] = attr_df[name].map(replacements).fillna(attr_df[name])

        self.image_dir   = image_dir
        self.attr_file   = attr_file

        # Get the list of image file names
        self.all_images   = os.listdir(image_dir)
        self.total_images = natsorted(self.all_images)

        # Convert attributes to a numpy array
        self.y           = attr_df.to_numpy()

        self.transform   = transform

    def __getitem__(self, index):
        img_loc = os.path.join(self.image_dir, self.total_images[index])
        img     = Image.open(img_loc).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        attrs   = self.y[index]
        attrs_t = torch.torch.tensor(attrs, dtype=torch.float32)

        return img, attrs_t

    def __len__(self):
        # returns the number of images in the Dataset
        return len(self.total_images)

### (iii) Create training, validation and test datasets

In [ ]:
train_data = CelebaDataset(
    attr_file,
    train_dir,
    transform = train_transforms
)

val_data = CelebaDataset(
    attr_file,
    val_dir,
    transform = val_transforms
)

test_data = CelebaDataset(
    attr_file,
    test_dir,
    transform = val_transforms
)

print(len(train_data))
print(len(val_data))
print(len(test_data))

In [ ]:
# Visualize some samples from the created training set

# Indexing the 'train_data' Dataset to get a single (image, label) sample

img , attrs  = train_data[0]

print(f'Image tensor:\n{img}')
print(f'Image shape: {img.shape}')
print(f'Image data type: {img.dtype}')
print(f'Attributes:\n{attrs}')
print(f'Attributes shape: {attrs.shape}')
print(f'Attributes type: {attrs.dtype}')

In [ ]:
# rearrange the order of dimensions on image tensor for using matplot lib
img_permuted = img.permute(1, 2, 0) # C,H,W --> H,W,C

# print shapes
print(f'Original image shape (pytorch tensor):   {img.shape}')
print(f'Permute image shape (matplotlib format): {img_permuted.shape}')

# plot the image

plt.figure(figsize=(10,7))
plt.imshow(img_permuted)
plt.axis(False)
plt.title(f'An image from training set')

### (iv) Convert the `Datasets` into `torch.utils.data.DataLoaders`

A DataLoader helps us to iterate trough the dataset em get a batch of samples each time.

Relevant configuration parameters:

```python
config["data_params"]["train_batch_size"]
config["data_params"]["val_batch_size"]
config["data_params"]["test_batch_size"]
config["data_params"]["num_workers"]
config["data_params"]["pin_memory"]
```

In [ ]:
# Convert each 'Dataset' into a 'DataLoader'

from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    dataset     = train_data,
    batch_size  = config["data_params"]["train_batch_size"],
    shuffle     = True,
    num_workers = config["data_params"]["num_workers"],
    pin_memory  = config["data_params"]["pin_memory"]
)

val_dataloader = DataLoader(
    dataset     = val_data,
    batch_size  = config["data_params"]["val_batch_size"],
    shuffle     = False,
    num_workers = config["data_params"]["num_workers"],
    pin_memory  = config["data_params"]["pin_memory"]
)

test_dataloader = DataLoader(
    dataset     = test_data,
    batch_size  = config["data_params"]["test_batch_size"],
    shuffle     = False,
    num_workers = config["data_params"]["num_workers"],
    pin_memory  = config["data_params"]["pin_memory"]
)

In [ ]:
print(train_dataloader)
print(val_dataloader)
print(test_dataloader)

print(len(train_dataloader)) # returns the number of batches in a training epoch
print(len(val_dataloader))   # returns the number of batches in a validation epoch
print(len(test_dataloader))  # returns the number of batches in a test epoch

In [ ]:
# print metadata about a sample retrieved from the training DataLoader

imgs, attrs = next(iter(train_dataloader))

print(f'Batch of images shape: {imgs.shape}')        # BS, Ch, H, W
print(f'Batch of attributes shape: {attrs.shape}')   # BS, 40

## The CVAE Model

In [ ]:
class ConditionalVAE(nn.Module):

    def __init__(self,
                 in_channels:    int,
                 num_attributes: int,
                 latent_dim:     int,
                 hidden_dims:    List = None,
                 img_size:       int  = 128,
                 **kwargs) -> None:
        super(ConditionalVAE, self).__init__()

        self.latent_dim = latent_dim
        self.img_size   = img_size

        # Layer that creates an embedding for the condition input 'y'
        self.embed_attrs = nn.Linear(
            in_features  = num_attributes,
            out_features = img_size * img_size
        )
        
        # Layer that creates the embedding for the data input 'x'
        self.embed_data  = nn.Conv2d(
            in_channels, 
            in_channels, 
            kernel_size=1
        )

        modules = []
        if hidden_dims is None:
            hidden_dims = [16, 32, 64, 128, 128]

        in_channels += 1 # To account for the extra label channel
        
         # Build the Encoder ...........................................
        
        for h_dim in hidden_dims:
            modules.append(
                nn.Sequential(
                    nn.Conv2d(
                        in_channels,
                        out_channels = h_dim,
                        kernel_size  = 3,
                        stride       = 2, 
                        padding      = 1
                    ),
                    nn.BatchNorm2d(h_dim),
                    nn.LeakyReLU())
            )
            in_channels = h_dim

        # Add the 5 blocks to the encoder model
        self.encoder = nn.Sequential(*modules)

        # Fully connected layer that outputs the latent mean
        self.fc_mu   = nn.Linear(
            in_features  = hidden_dims[-1]*16,
            out_features = latent_dim
        )

        # Fully connected layer that outputs the latent log-variance
        self.fc_var  = nn.Linear(
            in_features  = hidden_dims[-1]*16, 
            out_features = latent_dim
        )


        # Build the Decoder ...........................................
        modules = []

        # Fully connected layer to make the latent variables to have the
        # correct shape for decoder
        self.decoder_input = nn.Linear(
            in_features  = latent_dim + num_attributes,
            out_features = hidden_dims[-1] * 16
        )

        # hidden_dims = [128, 128, 64, 32, 16]
        hidden_dims.reverse()

        # Define 4 blocks composed of: ConvTranspose2d -> BatchNorm2d -> LeakyReLU
        for i in range(len(hidden_dims) - 1):
            modules.append(
                nn.Sequential(
                    nn.ConvTranspose2d(
                        hidden_dims[i],
                        hidden_dims[i + 1],
                        kernel_size    = 3,
                        stride         = 2,
                        padding        = 1,
                        output_padding = 1
                    ),
                    nn.BatchNorm2d(hidden_dims[i + 1]),
                    nn.LeakyReLU())
            )

        # Add the 4 blocks to the decoder model
        self.decoder = nn.Sequential(*modules)

        # Add a final block to the decoder composed of:
        #   ConvTranspose2d -> BatchNorm2d -> LeakyReLU -> Conv2d -> Tanh
        self.final_layer = nn.Sequential(
            nn.ConvTranspose2d(
                hidden_dims[-1],
                hidden_dims[-1],
                kernel_size    = 3,
                stride         = 2,
                padding        = 1,
                output_padding = 1
            ),
            nn.BatchNorm2d(hidden_dims[-1]),
            nn.LeakyReLU(),
            nn.Conv2d(
                hidden_dims[-1], 
                out_channels = 3,
                kernel_size  = 3, 
                padding      = 1
            ),
            nn.Sigmoid()
        )

    def encode(self, input: Tensor) -> List[Tensor]:
        """
        Encodes the input by passing it through the encoder network
        and returns the latent mean and log-variance.

        Arguments:
            * input: Input tensor to encoder [N x C x H x W]
        Returns:
            A list containing the mean and the log-variance of latent distribution.
        """

        # Passes 'input' through the encoder
        result = self.encoder(input)

         # Flatten the encoder output
        result = torch.flatten(result, start_dim=1)

        # Split the result into the mean and variance
        # of the latent Normal distribution
        mu      = self.fc_mu(result)
        log_var = self.fc_var(result)

        return [mu, log_var]

    def decode(self, z: Tensor) -> Tensor:
        """
        Maps the given latent variable 'z' back to the image space.

        Arguments:
            * z: (Tensor) [B x D]
        Returns:
            * The image generated by the decoder,
              a tensor with shape [B x C x H x W].
        """

        # Reshape latent 'z' with the fully connected layer
        result = self.decoder_input(z)
        # Reorganize the latent dimensions
        result = result.view(-1, 128, 4, 4)
        # Pass data through the 4 decoder blocks
        result = self.decoder(result)
        # Pass data through the final block
        result = self.final_layer(result)
        return result

    def reparameterize(self, mu: Tensor, logvar: Tensor) -> Tensor:
        """
        Reparameterization trick to allow the gradient flow through
        the latent sampler. In this way, sampling 'z' from N(mu, var) is
        modified to sampling 'eps' from N(0,1) followed by the definition
        z = mu + eps * e^logvar.

        Arguments:
            * mu: Mean of the latent Normal distribution, a tensor [B x D]
            * logvar: Log-variance of the latent Normal distribution, a tensor [B x D]
        Returns:
            The latent variable 'z', a tensor [B x D].
        """

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return eps * std + mu

    def forward(self, input: Tensor, **kwargs) -> List[Tensor]:
        '''
        Defines the forward pass through the CVAE: 
        encoder -> reparameterization -> decoder

        Returns:
            A list containing: x', x, mu, log_var.
        '''
        # The condition input 'y'
        y = kwargs['attributes'].float()

        # Create the embedding of the condition input 'y'
        embedded_attrs = self.embed_attrs(y)

        # Reorganize the condition embedding to be compatible with the data embedding
        embedded_attrs = embedded_attrs.view(-1, self.img_size, self.img_size).unsqueeze(1)

        # Create the embedding of the data input 'x'
        embedded_input = self.embed_data(input)

        # Concatenate condition and data embeddings along dimension 1
        x = torch.cat([embedded_input, embedded_attrs], dim = 1)

        # Pass x||y through the encoder  
        mu, log_var = self.encode(x)

        # Apply the reparameterization trick to the latent variables
        z = self.reparameterize(mu, log_var)

        # Concatenate condition embedding and latent variables 'z' along dimension 1
        z = torch.cat([z, y], dim = 1)

        # Pass z||y through the decoder 
        # and return x', x, mu, log_var 
        return  [self.decode(z), input, mu, log_var]

    def loss_function(
        self,
        reconst_img: Tensor,
        input_img:   Tensor,
        mu:          Tensor,
        log_var:     Tensor,
        kld_weight:  float ) -> dict:
        '''
        Computes the CVAE loss = reconstruction_loss + KLD_loss * kld_weight

        Arguments:
            * args
            * kwargs
        Returns:
            A tuple containing (loss, reconstruction term of loss, KLD term of loss)
        '''
        reconst_img  = reconst_img
        input_img    = input_img
        mu           = mu
        log_var      = log_var
        kld_weight   = kld_weight # Account for the minibatch samples from the dataset

        # Reconstruction term of the loss: SUM (x-x_reconstructed)^2
        reconst_loss = F.mse_loss(reconst_img, input_img, reduction='sum')

        # KL Divergence term of the loss
        #  D_KL(Normal(mu, var) || Normal(0, 1)) = -0.5 * [ log (var) + 1 - var^2 - mu^2 ]
        kld_loss = -0.5 * torch.sum(1 + log_var - mu ** 2 - log_var.exp())

        # Loss = -ELBO
        batch_size    = mu.shape[0]
        reconst_loss /= batch_size
        kld_loss     /= batch_size
        loss          = reconst_loss + kld_weight * kld_loss

        # print(f'Loss shape: {loss.shape} -> {loss}')

        return loss, reconst_loss, kld_loss

    def sample(self,
               num_samples: int,
               device:      int,
               **kwargs ) -> Tensor:
        """
        Samples the latent space and returns the corresponding reconstructed image.

        Arguments:
            * num_samples: Number of samples (an Int)
            * current_device: Device to run the model (an Int)
        Returns:
            The recontructed images given the samples (a tensor).
        """

        # input condition 'y'
        y = kwargs['attributes']
        y = y.to(device)

        # Draw 'num_samples' samples from the standard Normal N(0,1)
        z = torch.randn(num_samples, self.latent_dim)

        # Put the samples in the same device of the model
        z = z.to(device)

        # Concatenate condition embedding and latent variables 'z' along dimension 1
        z = torch.cat([z, y], dim=1)

        # Reconstruct 'num_samples' images from the samples||y that we draw
        samples = self.decode(z)

        return samples

    def generate(self, x: Tensor, **kwargs) -> Tensor:
        """
        Given input images 'x' and condition 'y', passes them through the CVAE 
        and returns the reconstructed images.

        Arguments:
            * x: Input images (a tensor [B x C x H x W]).
        Returns:
            The reconstructed images (a tensor [B x C x H x W]).
        """

        return self.forward(x, **kwargs)[0]

In [ ]:
model = ConditionalVAE(**config['model_params']).to(device)

model

In [ ]:
from torchinfo import summary

myconds = torch.randn(
    config["data_params"]["train_batch_size"],
    config["model_params"]["num_attributes"]
).to(device)

summary(model, attributes=myconds, input_size=(16, 3, 128, 128))

To test our model let us do a single forward pass (pass a sample batch from the training set through the model).

In [ ]:
# 1. Get a batch of images from the training DataLoader

img_batch , attrs_batch = next(iter(train_dataloader))

# 2. Get a single image from the batch and unsqueeze the image
#    so its shape fits the model

img_single = img_batch[0].unsqueeze(dim=0)
print(f"Single image shape: {img_single.shape}\n")

attrs_single = attrs_batch[0].unsqueeze(dim=0)
print(f"Single attributes shape: {attrs_single.shape}\n")

# 3. Perform a forward pass on a single image

model.eval()
with torch.inference_mode():
    img_out, img_in, mu_out, logvar_out = model(
        img_single.to(device),
        attributes=attrs_single.to(device)
    )

# 4. Print output

print(f"Img output:\n{img_out}\n")
print(f"Mean:\n{mu_out}\n")
print(f"LogVar:\n{logvar_out}\n")

## Creating `train_step` and `val_step` functions and `train` to combine them  

Let us start by making `train_step()`.

In [ ]:
def train_step(
    model:      torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn:    torch.nn.Module,
    optimizer:  torch.optim.Optimizer,
    kld_weight: float,
    epoch_id:   int,
    device:     torch.device,
    ) -> Tuple[float, float, float]:
    """
    Trains a PyTorch model for a single epoch.

    Turns a target PyTorch model to training mode and then
    runs through all of the required training steps (forward
    pass, loss calculation, optimizer step).

    Arguments:
        model:      A PyTorch model to be trained.
        dataloader: A DataLoader instance for the model to be trained on.
        loss_fn:    A PyTorch loss function to minimize.
        optimizer:  A PyTorch optimizer to help minimize the loss function.
        kld_weight: Weight to apply to the KLD term of the loss.
        device:     A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
        A tuple of training loss, reconstruction loss, kld loss.
        In the form (train_loss, reconst_loss, kld_loss). For example:

        (0.3456, 0.8743, 0.5678)
    """

    # Put the model in train mode
    model.train()

    # Setup train loss, reconstruction loss, and KLD loss values
    train_loss         = 0.
    train_reconst_loss = 0.
    train_kld_loss     = 0.

    # Loop through data loader data batches
    for batch, (X, attrs) in enumerate(dataloader):
        # Send data to target device
        X     = X.to(device)
        attrs = attrs.to(device)

        # 1. Forward pass
        X_out, _, mu_out, logvar_out = model(X, attributes=attrs)

        # 2. Calculate and accumulate the losses
        loss, reconst_loss, kld_loss = loss_fn(
            X_out,
            X,
            mu_out,
            logvar_out,
            kld_weight,
        )

        train_loss         += loss.item()
        train_reconst_loss += reconst_loss.item()
        train_kld_loss     += kld_loss.item()

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

    # Adjust losses to get average value per batch
    train_loss         = train_loss / len(dataloader)
    train_reconst_loss = train_reconst_loss / len(dataloader)
    train_kld_loss     = train_kld_loss / len(dataloader)

    return train_loss, train_reconst_loss, train_kld_loss

Now we will implement `val_step()`.

In [ ]:
def val_step(
    model:      torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn:    torch.nn.Module,
    kld_weight: float,
    epoch_id:   int,
    device:     torch.device) -> Tuple[float, float, float]:
    """
    Runs a validation step with a PyTorch model for a single epoch.

    Turns a target PyTorch model to "eval" mode and then performs
    a forward pass on a testing dataset.

    Arguments:
        model:      A PyTorch model to be tested.
        dataloader: A DataLoader instance for the model to be tested on.
        loss_fn:    A PyTorch loss function to calculate loss on the test data.
        kld_weight: Weight to apply to the KLD term of the loss.
        device:     A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
        A tuple of validation loss, reconstruction loss, kld loss.
        In the form (train_loss, reconst_loss, kld_loss). For example:

        (0.3456, 0.8743, 0.5678)
    """
    # Put model in eval mode
    model.eval()

    # Setup validation loss, reconstruction loss, and KLD loss values
    val_loss         = 0.
    val_reconst_loss = 0.
    val_kld_loss     = 0.

    # Turn on inference context manager
    with torch.inference_mode():

        # Loop through DataLoader batches
        for batch, (X, attrs) in enumerate(dataloader):

            # Send data to target device
            X     = X.to(device)
            attrs = attrs.to(device)

            # 1. Forward pass
            X_out, _, mu_out, logvar_out = model(X, attributes=attrs)

            # 2. Calculate and accumulate losses
            loss, reconst_loss, kld_loss = loss_fn(
                X_out,
                X,
                mu_out,
                logvar_out,
                kld_weight,
            )

            val_loss         += loss.item()
            val_reconst_loss += reconst_loss.item()
            val_kld_loss     += kld_loss.item()

    # Adjust losses to get average value per batch
    val_loss         = val_loss / len(dataloader)
    val_reconst_loss = val_reconst_loss / len(dataloader)
    val_kld_loss     = val_kld_loss / len(dataloader)

    return val_loss, val_reconst_loss, val_kld_loss

And we will combine `train_step()` and `val_step()` into `train()`.

In [ ]:
from tqdm.auto import tqdm

def train(
    model: torch.nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    val_dataloader:   torch.utils.data.DataLoader,
    optimizer:        torch.optim.Optimizer,
    loss_fn:          torch.nn.Module,
    kld_weight:       float,
    epochs:           int,
    device:           torch.device ) -> Dict[str, List[float]]:
    """
    Trains and validates a PyTorch model.

    Passes a target PyTorch models through train_step() and val_step()
    functions for a number of epochs, training and validating the model
    in the same epoch loop.

    Calculates, prints and stores evaluation metrics throughout.

    Arguments:
        model:            A PyTorch model to be trained and tested.
        train_dataloader: A DataLoader instance for the model to be trained on.
        val_dataloader:   A DataLoader instance for the model to be tested on.
        optimizer:        A PyTorch optimizer to help minimize the loss function.
        loss_fn:          A PyTorch loss function to calculate loss on both datasets.
        kld_weight:       Weight to apply to the KLD term of the loss.
        epochs:           An integer indicating how many epochs to train for.
        device:           A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
        A dictionary of training and validation losses. Each metric has a value 
        in a list for each epoch.
        In the form:
            {
            train_loss:         [...],
            train_reconst_loss: [...],
            train_kld_loss:     [...],
            val_loss:           [...],
            val_reconst_loss:   [...],
            val_kld_loss:       [...]
            }
        For example if training for epochs=2:
            {
            train_loss:         [2.0616, 1.0537],
            train_reconst_loss: [0.3945, 0.4945],
            train_kld_loss:     [0.5945, 0.6945],
            val_loss:           [1.2641, 1.5706],
            val_reconst_loss:   [0.7400, 0.8973],
            val_kld_loss:       [0.9945, 0.0545],
            }
    """

    # Create empty results dictionary
    results = {
        "train_loss":         [],
        "train_reconst_loss": [],
        "train_kld_loss":     [],
        "val_loss":           [],
        "val_reconst_loss":   [],
        "val_kld_loss":       []
    }

    # Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):

        train_loss, train_rec_loss, train_kld_loss = train_step(
            model      = model,
            dataloader = train_dataloader,
            loss_fn    = loss_fn,
            optimizer  = optimizer,
            kld_weight = kld_weight,
            epoch_id   = epoch,
            device     = device,
        )

        val_loss, val_rec_loss, val_kld_loss = val_step(
            model      = model,
            dataloader = test_dataloader,
            loss_fn    = loss_fn,
            kld_weight = kld_weight,
            epoch_id   = epoch,
            device     = device
        )

        # Print out what's happening
        print(
            f"Epoch: {epoch+1} | "
            f"Losses-> train: {train_loss:.4f} | "
            f"train_rec: {train_rec_loss:.4f} | "
            f"train_kld: {train_kld_loss:.4f} | "
            f"val: {val_loss:.4f} | "
            f"val_rec: {val_rec_loss:.4f} | "
            f"val_kld: {val_kld_loss:.4f}"
        )

        # Update results dictionary
        results["train_loss"].append(train_loss)
        results["train_reconst_loss"].append(train_rec_loss)
        results["train_kld_loss"].append(train_kld_loss)
        results["val_loss"].append(val_loss)
        results["val_reconst_loss"].append(val_rec_loss)
        results["val_kld_loss"].append(val_kld_loss)

        # Log metrics to W&B
        wandb.log(
            {
            "train_loss":     train_loss,
            "train_rec_loss": train_rec_loss,
            "train_kld_loss": train_kld_loss,
            "val_loss":       val_loss,
            "val_rec_loss":   val_rec_loss,
            "val_kld_loss":   val_kld_loss,
            "epoch":          epoch,
            }
        )

    # Return the filled results at the end of the epochs
    return results

## Create a function to save the model

Let us setup a function to save our model to a directory.

In [ ]:
from pathlib import Path

def save_model(
    model: torch.nn.Module,
    target_dir: str,
    model_name: str ):
    """
    Saves a PyTorch model to a target directory.

    Args:
        model:      A target PyTorch model to save.
        target_dir: A directory for saving the model to.
        model_name: A filename for the saved model. Should include
                    either ".pth" or ".pt" as the file extension.

    Example of usage:
        save_model(
            model=model_0,
            target_dir="models",
            model_name="05_going_modular_tingvgg_model.pth"
        )
    """
    # Create target directory
    target_dir_path = Path(target_dir)
    target_dir_path.mkdir(parents=True, exist_ok=True)

    # Create model save path
    assert model_name.endswith(".pth") or model_name.endswith(".pt"), "model_name should end with '.pt' or '.pth'"
    model_save_path = target_dir_path / model_name

    # Save the model state_dict()
    print(f"[INFO] Saving model to: {model_save_path}")
    torch.save(
        obj = model.state_dict(),
        f   = model_save_path
    )

## Train, validate and save the model

Let us leverage the functions we have got above to train, validate and save a model to file.

In [ ]:
# Set random seeds

torch.manual_seed(config["exp_params"]["manual_seed"])
torch.cuda.manual_seed(config["exp_params"]["manual_seed"])

# Recreate an instance of VanillaVAE

model_vae = ConditionalVAE(**config['model_params']).to(device)

# Setup loss function and optimizer

loss_fn = model_vae.loss_function

optimizer = torch.optim.Adam(
    params = model_vae.parameters(), 
    lr     = config["exp_params"]["LR"]
)

# Tell wandb to watch what the model gets up to: gradients, weights, and more
#
#wandb.watch(model_vae, loss_fn, log="all", log_freq=10)

# Start the timer

from timeit import default_timer as timer
start_time = timer()

# Train model_vae

vae_results = train(model=model_vae,
    train_dataloader = train_dataloader,
    val_dataloader   = val_dataloader,
    optimizer        = optimizer,
    loss_fn          = loss_fn,
    kld_weight       = config["exp_params"]["kld_weight"],
    epochs           = config["exp_params"]["epochs"],
    device           = device
)

# End the timer and print out how long it took

end_time = timer()
print(f"[INFO] Total training time: {end_time-start_time:.3f} seconds")

# Save the model to file

save_model(model=model_vae,
    target_dir = "models",
    model_name = f"{BASE_FILE_NAME}.pth"
)

## Save results to a CSV file

Convert the `vae_results` dictionary, containing list with the loss values, to a DataFrame and later to a CSV file.

In [ ]:
vae_results_df = pd.concat({k: pd.Series(v) for k, v in vae_results.items()}, axis=1)

vae_results_df.to_csv(
    f"results/{BASE_FILE_NAME}_results.csv",
    sep=',',
    index=False,
    encoding='utf-8'
)

## Plot the training and validation losses 

In [ ]:
vae_results_df[["train_loss", "train_reconst_loss", "train_kld_loss"]].plot()
vae_results_df[["val_loss", "val_reconst_loss", "val_kld_loss"]].plot()

## Reconstruct images drawn from the test set and generate new images given samples from the latent distribution

In [ ]:

from torchvision.utils import save_image

def reconstruct_sample_images(
    model:             torch.nn.Module,
    test_dataloader:   torch.utils.data.DataLoader,
    attributes:        torch.Tensor,
    file_inputs:       Union[str, Path],
    file_reconstructs: Union[str, Path],
    file_samples:      Union[str, Path],
    device:            torch.device
    ):
    '''
    Get a batch of images from test DataLoader, pass them through the model 
    and save the reconstructed images to file.
    Get 144 samples from latent distribution, generate the correspondent images
    and save the images to file.

    Arguments:
        model:             The PyTorch model. 
        test_dataloader:   The test DataLoader to get the images that 
                           will be used in reconstruction.
        attributes:        Attributes to condition the image generation.
        file_inputs:       File/Path to save the batch of input images used in reconstruction.
        file_reconstructs: File/Path to save the batch of reconstructed images.
        file_samples:      File/Path to save the 144 generated images.
        device:            A target device where the model is located (e.g. "cuda" or "cpu").
    '''
    # Get a batch of images from the test Dataloader that will be used in reconstruction
    input_img, input_attrs = next(iter(test_dataloader))

    # Save the batch of images retrieved from the test Dataloader to a file
    save_image(
        input_img.data,
        file_inputs,
        normalize = True,
        nrow      = 12
    )

    # Pass the batch of images through the model
    input_img   = input_img.to(device)
    input_attrs = input_attrs.to(device)
    recons_img  = model.generate(input_img, attributes=input_attrs)

    # Save the reconstructed images to a file
    save_image(
        recons_img.cpu().data,
        file_reconstructs,
        normalize = True,
        nrow      = 12
    )

    try:
        # Get 144 samples from the latent distribution
        # and pass them through the decoder <=> generate images
        generated_img = model.sample(144, attributes=attributes, device=device)

        # Save the generated images to a file
        save_image(
            generated_img.cpu().data,
            file_samples,
            normalize=True,
            nrow=12
        )
    except Warning:
        pass


## Conditioned Image Generation through the image attributes

### Attributes of CelebA Images
| Code | Attribute        | Code | Attribute        | Code | Attribute            | Code | Attribute        |
| :--- | :--------------- | :--- | :--------------- | :--- | :------------------- | :--- | :--------------- | 
| 1    | 5_o_Clock_Shadow | 11   | Blurry           | 21   | Male                 | 31   | Sideburns        | 
| 2    | Arched_Eyebrows  | 12   | Brown_Hair       | 22   | Mouth_Slightly_Open  | 32   | Smiling          | 
| 3    | Attractive       | 13   | Bushy_Eyebrows   | 23   | Mustache             | 33   | Straight_Hair    | 
| 4    | Bags_Under_Eyes  | 14   | Chubby           | 24   | Narrow_Eyes          | 34   | Wavy_Hair        | 
| 5    | Bald             | 15   | Double_Chin      | 25   | No_Beard             | 35   | Wearing_Earrings | 
| 6    | Bangs            | 16   | Eyeglasses       | 26   | Oval_Face            | 36   | Wearing_Hat      | 
| 7    | Big_Lips         | 17   | Goatee           | 27   | Pale_Skin            | 37   | Wearing_Lipstick | 
| 8    | Big_Nose         | 18   | Gray_Hair        | 28   | Pointy_Nose          | 38   | Wearing_Necklace | 
| 9    | Black_Hair       | 19   | Heavy_Makeup     | 29   | Receding_Hairline    | 39   | Wearing_Necktie  | 
| 10   | Blond_Hair       | 20   | High_Cheekbones  | 30   | Rosy_Cheeks          | 40   | Young            | 


### Used attributes

|                  |                  |                  |
| :--------------- | :--------------- | :--------------- |
| Attractive  ( 3) | Black_Hair  ( 9) | Blond_Hair  (10) |
| Brown_Hair  (12) | Eyeglasses  (16) | Goatee      (17) |
| Gray_Hair   (18) | male        (21) | Mustache    (23) |
| No_Beard    (25) | Pale_Skin   (27) | Rosy_Cheeks (30) |
| Smiling     (32) | Young       (40) | Brown_Hair  (12) |

### Created profiles

| Profile 1        | Profile 2        | Profile 3        | Profile 4        | Profile 5        | Profile 6        |
| :--------------- | :--------------- | :--------------- | :--------------- | :--------------- | :--------------- |
| Attractive  ( 3) | Attractive  ( 3) | Blond_Hair  (10) | Attractive  ( 3) | Attractive  ( 3) | Eyeglasses  (16) |
| Black_Hair  ( 9) | Gray_Hair   (18) | Eyeglasses  (16) | Brown_Hair  (12) | Black_Hair  ( 9) | Gray_Hair   (18) |
| No_Beard    (25) | No_Beard    (25) | No_Beard    (25) | Male        (21) | Male        (21) | Male        (21) |
| Rosy_Cheeks (30) | Smiling     (32) | Pale_Skin   (27) | Pale_Skin   (27) | Mustache    (23) | No_Beard    (25) |
| Young       (40) |                  | Rosy_Cheeks (30) | Rosy_Cheeks (30) | No_Beard    (25) | Smiling     (32) |
|                  |                  | Young       (40) | Smiling     (32) | Pale_Skin   (27) |                  |
|                  |                  |                  |                  | Young       (40) |                  |


In [ ]:
#1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40
attrs_0_5 = torch.tensor(
[
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
[0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
],
dtype=torch.float32
)

In [ ]:
reconstruct_sample_images(
    model_vae,
    test_dataloader,
    attributes        = attrs_0_5,
    file_inputs       = f"results/{BASE_FILE_NAME}_input.jpg",
    file_reconstructs = f"results/{BASE_FILE_NAME}_reconstructed.jpg",
    file_samples      = f"results/{BASE_FILE_NAME}_generated.jpg",
    device            = device,
)

In [ ]:
def reconstruct_sample_images_v2(
    model:             torch.nn.Module,
    test_dataloader:   torch.utils.data.DataLoader,
    attributes:        torch.Tensor,
    file_inputs:       Union[str, Path],
    file_reconstructs: Union[str, Path],
    file_samples:      Union[str, Path],
    nrows:             int,
    ncols:             int,
    device:            torch.device
    ):
    '''
    Get a batch of images from test DataLoader and pass them through the model.
    Among the batch of input images, save a grid of nrows x ncols images to a file.
    Among the batch of reconstructed images, save a grid of nrows x ncols images to a file.
    Get 144 samples from latent distribution, generate the correspondent images
    and save a grid of nrows x ncols generated images to file.

    Arguments:
        model:             The PyTorch model. 
        test_dataloader:   The test DataLoader to get the images that 
                           will be used in reconstruction.
        attributes:        Attributes to condition the image generation.
        file_inputs:       File/Path to save the batch of input images used in reconstruction.
        file_reconstructs: File/Path to save the batch of reconstructed images.
        file_samples:      File/Path to save the 144 generated images.
        nrows:             Number of rows of images to save to the files.
        ncols:             Number of columns of images to save to the files.
        device:            A target device where the model is located (e.g. "cuda" or "cpu").
    '''
    # Get a batch of images from the test Dataloader that will be used in reconstruction
    input_img, input_attrs = next(iter(test_dataloader))

    assert nrows*ncols <= len(input_img), f"{nrows*ncols} must be less or equal to the batch size ({len(input_img)})"

    # Save the batch of images retrieved from the test Dataloader to a file
    save_image(
        input_img[0:nrows*ncols].data,
        file_inputs,
        normalize = True,
        nrow      = ncols,
    )

    # Pass the batch of images through the model
    input_img   = input_img.to(device)
    input_attrs = input_attrs.to(device)
    recons_img  = model.generate(input_img, attributes=input_attrs)

    # Save the reconstructed images to a file
    save_image(
        recons_img[0:nrows*ncols].cpu().data,
        file_reconstructs,
        normalize = True,
        nrow      = ncols,
    )

    try:
        # Get 144 samples from the latent distribution 
        # and pass them through the decoder <=> generate images
        generated_img = model.sample(144, attributes=attributes, device=device)

        # Save the generated images to a file
        save_image(
            generated_img[0:nrows*ncols].cpu().data, 
            file_samples,
            normalize=True,
            nrow=ncols,
        )
    except Warning:
        pass

In [ ]:
reconstruct_sample_images_v2(
    model_vae,
    test_dataloader,
    attributes        = attrs_0_5,
    file_inputs       = f"results/{BASE_FILE_NAME}_input_6x6.jpg",
    file_reconstructs = f"results/{BASE_FILE_NAME}_reconstructed_6x6.jpg",
    file_samples      = f"results/{BASE_FILE_NAME}_generated_6x6.jpg",
    nrows             = 6,
    ncols             = 6,
    device            = device,
)